In [94]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Setup

In [95]:
import os

os.chdir(r"C:\Kaggle_Competition\Playground\S6E6-STELLAR-CLASS")
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import json
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss, balanced_accuracy_score
from sklearn.inspection import permutation_importance
import mlflow
from tqdm.notebook import tqdm
import itertools
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    TargetEncoder,
    OneHotEncoder,
    StandardScaler,
    LabelEncoder,
    KBinsDiscretizer,
    RobustScaler
)
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.base import TransformerMixin, BaseEstimator
from feature_engine.outliers import Winsorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from lightgbm import LGBMClassifier
from lightgbm import early_stopping, log_evaluation
from xgboost import XGBClassifier
from xgboost.callback import EarlyStopping
from catboost import CatBoostClassifier
from pytabkit import (
    RealMLP_TD_Classifier,
    TabM_D_Classifier,
    Resnet_RTDL_D_Classifier,
    TabM_HPO_Classifier,
    RealMLP_HPO_Classifier,
    FTT_D_Classifier,
)
from sklearn.manifold import TSNE
import copy
import tabpfn_client
from tabpfn_client import TabPFNClassifier
from autogluon.tabular import TabularPredictor
pd.set_option("display.max_columns", None)

from dotenv import load_dotenv
import shap
from shap.explainers import LinearExplainer
import config
import warnings
warnings.filterwarnings("ignore")

In [96]:
load_dotenv(r'.env')

api_key = os.getenv('api_key')

In [97]:
train = pd.read_csv("data/raw/train.csv")
test = pd.read_csv("data/raw/test.csv")
orig = pd.read_csv("data/orig/star_classification.csv")

train_ids = train[config.ID_COL].values
test_ids = test[config.ID_COL].values

print(f"Shape of train data: {train.shape}")
print(f"Shape of test data: {test.shape}")
print(f"Shape of orig data: {orig.shape}")

Shape of train data: (577347, 12)
Shape of test data: (247435, 11)
Shape of orig data: (100000, 18)


In [98]:
# Quantitative features
num_cols = ["alpha", "delta", "u", "g", "r", "i", "z", "redshift"]

# Qualitative features
cat_cols = ["spectral_type", "galaxy_population"]

# Target column
target = "class"

In [99]:
orig2 = pd.read_csv(r'data\orig\StarClassificationDataset.csv')
orig2 = orig2.dropna(axis=0)
# Create the structural mapping dictionary
filter_mapping = {
    'UV_filter': 'u',
    'green_filter': 'g',
    'red_filter': 'r',
    'near_IR_filter': 'i',
    'IR_filter': 'z',
    'red_shift': 'redshift',
    'object_ID': 'obj_ID',
    'plate_ID': 'plate'
}

# Rename the columns in your dataframe
orig2 = orig2.rename(columns=filter_mapping)
orig2 = orig2.drop(index=79543).reset_index(drop=True)

In [100]:
orig = pd.concat([orig, orig2], axis=0).reset_index(drop=True)

In [101]:
# Train data
X = train.drop(["id", config.TARGET_COL], axis=1)
y = train[config.TARGET_COL]

# Orig data
orig = orig.drop(index=79543).reset_index(drop=True)

def spectral_type(g, r):
    return pd.cut(
        r - g, 
        [-np.inf, -1, -0.5, 0, np.inf],
        labels=['M', 'G/K', 'A/F', 'O/B']
    ).astype(str)

def galaxy_population(u, r):
    return pd.cut(
        u - r, 
        [-np.inf, 1.4, 2.2, np.inf],
        labels=['Blue_Cloud', 'Green_Valley', 'Red_Sequence']
    ).astype(str)

orig["spectral_type"] = spectral_type(
    orig["g"],
    orig["r"]
)

orig["galaxy_population"] = galaxy_population(
    orig["u"],
    orig["r"]
)

orig = orig.drop(index=100009).reset_index(drop=True)
X_orig = orig[num_cols + cat_cols]
X_orig['alpha'] = X_orig['alpha'].astype(float)
y_orig = orig[target]

# Test data
X_test = test.drop("id", axis=1)

enc = LabelEncoder()
y = enc.fit_transform(y)
y_orig = enc.transform(y_orig)

### Feature engineering

In [102]:
def fe(df):
    # Calculate consecutive color differences (adjacent bands)
    df["u_g"] = df["u"] - df["g"]
    df["g_r"] = df["g"] - df["r"]
    df["r_i"] = df["r"] - df["i"]
    df["i_z"] = df["i"] - df["z"]

    # Calculate crucial extended color differences
    df["u_r"] = df["u"] - df["r"]
    df["u_i"] = df["u"] - df["i"]

    df["redshift_g_r"] = df["redshift"] * df["g_r"]
    df["redshift_u_g"] = df["redshift"] * df["u_g"]

    alpha_rad = np.radians(df["alpha"])
    delta_rad = np.radians(df["delta"])

    df["coord_X"] = np.cos(delta_rad) * np.cos(alpha_rad)
    df["coord_Y"] = np.cos(delta_rad) * np.sin(alpha_rad)
    df["coord_Z"] = np.sin(delta_rad)

    df["coord_radial_dist"] = np.sqrt(df["alpha"] ** 2 + df["delta"] ** 2)

    # deviation from mean redshift
    df["redshift_dev_from_galaxy_population"] = df["redshift"] - df.groupby("galaxy_population")["redshift"].transform("mean")
    df["redshift_dev_from_spectral_type"] = df["redshift"] - df.groupby("spectral_type")["redshift"].transform("mean")

    # Color per distance
    for c in ["u_g", "g_r", "r_i", "i_z"]:
        df[f"{c}_per_redshift"] = (df[c] / (df["redshift"].abs() + 1e-6)).astype("float32")

    # Descriptive stats
    bands = ["u", "g", "r", "i", "z"]
    df["mag_std"] = df[bands].std(axis=1).astype("float32")
    df["mag_var"] = df[bands].var(axis=1).astype("float32")
    df["mag_skew"] = df[bands].skew(axis=1).astype("float32")
    df["mag_kurt"] = df[bands].kurt(axis=1).astype("float32")

    # category interaction
    df['galaxy_population_x_spectral_type'] = df['galaxy_population'] + "x" + df["spectral_type"]
    df['galaxy_population_x_spectral_type'] = df['galaxy_population_x_spectral_type'].astype('category')

    df['alpha+delta'] = (df['alpha'] + df['delta']).astype('float32')
    df['alpha-delta'] = (df['alpha'] - df['delta']).astype('float32')
    df['alpha%1'] = (df['alpha'] % 1).astype('float32')
    df['delta%1'] = (df['delta'] % 1).astype('float32')

    # Drop irrelevant cols
    df = df.drop(['galaxy_population', 'spectral_type'], axis=1)

    return df

X = fe(X)
X_test = fe(X_test)
X_orig = fe(X_orig)

In [103]:
num_cols = X.select_dtypes(include='number').columns.to_list()
cat_cols = X.select_dtypes(exclude='number').columns.to_list()

In [104]:
print(X.shape)
print(X_test.shape)
print(X_orig.shape)

(577347, 35)
(247435, 35)
(199988, 35)


### Logistic Regression

In [112]:
# Predictions
oof_predictions = np.zeros((len(train), config.N_CLASSES), dtype=np.float32)
test_predictions = np.zeros((len(X_test), config.N_CLASSES), dtype=np.float32)

# Experiment Setup
EXPERIMENT_NAME = "lr"
VERSION = "v2"
RUN_NAME = f"{EXPERIMENT_NAME}_{VERSION}_seed{config.SEED}"

mlflow.set_experiment(EXPERIMENT_NAME)

# CV
skf = StratifiedKFold(n_splits=config.N_FOLDS, shuffle=True, random_state=config.SEED)

fold_loglosses = []
fold_baccs = []

test_weight = 1 / config.N_FOLDS

with mlflow.start_run(run_name=RUN_NAME):
    for fold, (train_indices, valid_indices) in tqdm(
        enumerate(skf.split(X, y), start=1), desc="Model Training", total=config.N_FOLDS
    ):
        # Train and validation data
        X_train = X.iloc[train_indices]
        X_valid = X.iloc[valid_indices]

        y_train = y[train_indices]
        y_valid = y[valid_indices]

        # Preprocessor
        preprocess = ColumnTransformer(
            transformers=[
                #("encoder", OneHotEncoder(handle_unknown="ignore"), cat_cols),
                ("scaler", RobustScaler(), num_cols),
            ],
            remainder="passthrough",
        )

        # Fit and transform preprocessor 
        X_train_processed = preprocess.fit_transform(X_train, y_train)
        X_valid_processed = preprocess.transform(X_valid)
        X_test_processed = preprocess.transform(X_test)

        model = LogisticRegression(**config.LR_PARAMS)

        # Training model
        model.fit(X_train_processed, y_train)

        # Fold level predictions
        fold_oof_predictions = model.predict_proba(X_valid_processed)
        fold_test_predictions = model.predict_proba(X_test_processed)

        # Updating global predictions
        oof_predictions[valid_indices] = fold_oof_predictions
        test_predictions += fold_test_predictions * test_weight

        # Inference and scoring
        fold_logloss = log_loss(y_valid, fold_oof_predictions)
        fold_preds = model.predict(X_valid_processed)

        fold_bacc = balanced_accuracy_score(y_valid, fold_preds)

        fold_loglosses.append(fold_logloss)
        fold_baccs.append(fold_bacc)

        print(f"Fold: {fold} | LogLoss: {fold_logloss:.5f} | BACC: {fold_bacc:.5f}")

    # Calculating important metrices
    oof_logloss = log_loss(y, oof_predictions)
    oof_preds = model.classes_[np.argmax(oof_predictions, axis=1)]
    oof_bacc = balanced_accuracy_score(y, oof_preds)
    std_oof_logloss = np.std(fold_loglosses)
    std_oof_bacc = np.std(fold_baccs)

    # Mlflow metrics logging
    mlflow.log_metric("oof_logloss", oof_logloss)
    mlflow.log_metric("oof_bacc", oof_bacc)
    mlflow.log_metric("std_oof_logloss", std_oof_logloss)
    mlflow.log_metric("std_oof_bacc", std_oof_bacc)

    # Mlflow params logging
    mlflow.log_params(config.LR_PARAMS)

    # Mlflow input feature names logging
    mlflow.log_dict({"feature_names": list(X_train.columns)}, "feature_names.json")

    # Mlflow preprocessing steps logging
    mlflow.log_text(str(preprocess), "steps.txt")

    # Garbase collector
    if config.N_FOLDS < 5:
        del X_train, y_train, X_valid, y_valid, model, X_train_processed, X_valid_processed, X_test_processed

# Saving oof predictions
oof_proba_df = pd.DataFrame(oof_predictions, columns=enc.classes_)
oof_proba_df.insert(0, "id", train_ids)

oof_path = Path(config.OOF_PROBA_DIR, f"{RUN_NAME}_oof_proba.csv")
oof_proba_df.to_csv(oof_path, index=False)

# Saving test predictions
test_proba_df = pd.DataFrame(test_predictions, columns=enc.classes_)
test_proba_df.insert(0, "id", test_ids)

test_path = Path(config.TEST_PROBA_DIR, f"{RUN_NAME}_test_proba.csv")
test_proba_df.to_csv(test_path, index=False)

Model Training:   0%|          | 0/5 [00:00<?, ?it/s]

ValueError: could not convert string to float: 'Red_SequencexM'

In [ ]:
feature_names = preprocess.get_feature_names_out()

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_valid_processed_df = pd.DataFrame(
    X_valid_processed,
    columns=feature_names,
    index=X_valid.index
)

explainer = shap.LinearExplainer(
    model,
    X_train_processed_df # background data
)

shapley_values = explainer(
    X_valid_processed_df
)

In [ ]:
for i in range(3):
    shap.plots.bar(
        shapley_values[..., i]
    )

### HistGBM

In [ ]:
"""categorical_cols = ["spectral_type", "galaxy_population"]

X[categorical_cols] = X[categorical_cols].astype('category')
X_test[categorical_cols] = X_test[categorical_cols].astype('category')"""

In [105]:
# Predictions
oof_predictions = np.zeros((len(train), config.N_CLASSES), dtype=np.float32)
test_predictions = np.zeros((len(X_test), config.N_CLASSES), dtype=np.float32)

# Experiment Setup
EXPERIMENT_NAME = "histgbm"
VERSION = "v3"
RUN_NAME = f"{EXPERIMENT_NAME}_{VERSION}_seed{config.SEED}"

mlflow.set_experiment(EXPERIMENT_NAME)

# CV
skf = StratifiedKFold(n_splits=config.N_FOLDS, shuffle=True, random_state=config.SEED)

fold_loglosses = []
fold_baccs = []

test_weight = 1 / config.N_FOLDS

with mlflow.start_run(run_name=RUN_NAME):
    for fold, (train_indices, valid_indices) in tqdm(
        enumerate(skf.split(X, y), start=1), desc="Model Training", total=config.N_FOLDS
    ):
        # Train and validation data
        X_train = X.iloc[train_indices]
        X_valid = X.iloc[valid_indices]

        y_train = y[train_indices]
        y_valid = y[valid_indices]

        model = HistGradientBoostingClassifier(**config.HISTGBM_PARAMS)

        # Training model
        model.fit(X_train, y_train)

        # Fold level predictions
        fold_oof_predictions = model.predict_proba(X_valid)
        fold_test_predictions = model.predict_proba(X_test)

        # Updating global predictions
        oof_predictions[valid_indices] = fold_oof_predictions
        test_predictions += fold_test_predictions * test_weight

        # Inference and scoring
        fold_logloss = log_loss(y_valid, fold_oof_predictions)
        fold_preds = model.predict(X_valid)

        fold_bacc = balanced_accuracy_score(y_valid, fold_preds)

        fold_loglosses.append(fold_logloss)
        fold_baccs.append(fold_bacc)

        print(f"Fold: {fold} | LogLoss: {fold_logloss:.5f} | BACC: {fold_bacc:.5f}")

    # Calculating important metrices
    oof_logloss = log_loss(y, oof_predictions)
    oof_preds = model.classes_[np.argmax(oof_predictions, axis=1)]
    oof_bacc = balanced_accuracy_score(y, oof_preds)
    std_oof_logloss = np.std(fold_loglosses)
    std_oof_bacc = np.std(fold_baccs)

    # Mlflow metrics logging
    mlflow.log_metric("oof_logloss", oof_logloss)
    mlflow.log_metric("oof_bacc", oof_bacc)
    mlflow.log_metric("std_oof_logloss", std_oof_logloss)
    mlflow.log_metric("std_oof_bacc", std_oof_bacc)

    # Mlflow params logging
    mlflow.log_params(config.HISTGBM_PARAMS)

    # Mlflow input feature names logging
    mlflow.log_dict({"feature_names": list(X_train.columns)}, "feature_names.json")

    # Garbase collector
    if config.N_FOLDS < 5:
        del X_train, y_train, X_valid, y_valid, model

# Saving oof predictions
oof_proba_df = pd.DataFrame(oof_predictions, columns=enc.classes_)
oof_proba_df.insert(0, "id", train_ids)

oof_path = Path(config.OOF_PROBA_DIR, f"{RUN_NAME}_oof_proba.csv")
oof_proba_df.to_csv(oof_path, index=False)

# Saving test predictions
test_proba_df = pd.DataFrame(test_predictions, columns=enc.classes_)
test_proba_df.insert(0, "id", test_ids)

test_path = Path(config.TEST_PROBA_DIR, f"{RUN_NAME}_test_proba.csv")
test_proba_df.to_csv(test_path, index=False)

Model Training:   0%|          | 0/5 [00:00<?, ?it/s]

Fold: 1 | LogLoss: 0.09802 | BACC: 0.96510
Fold: 2 | LogLoss: 0.10166 | BACC: 0.96528
Fold: 3 | LogLoss: 0.10240 | BACC: 0.96391
Fold: 4 | LogLoss: 0.10045 | BACC: 0.96446
Fold: 5 | LogLoss: 0.10121 | BACC: 0.96455


### LightGBM

In [ ]:
# Predictions
oof_predictions = np.zeros((len(train), config.N_CLASSES), dtype=np.float32)
test_predictions = np.zeros((len(X_test), config.N_CLASSES), dtype=np.float32)

# Experiment Setup
EXPERIMENT_NAME = "lightgbm"
VERSION = "v6"
RUN_NAME = f"{EXPERIMENT_NAME}_{VERSION}_seed{config.SEED}"

mlflow.set_experiment(EXPERIMENT_NAME)

# CV
skf = StratifiedKFold(n_splits=config.N_FOLDS, shuffle=True, random_state=config.SEED)

fold_loglosses = []
fold_baccs = []

test_weight = 1 / config.N_FOLDS

with mlflow.start_run(run_name=RUN_NAME):
    for fold, (train_indices, valid_indices) in tqdm(
        enumerate(skf.split(X, y), start=1),
        desc="Model Training",
        total=config.N_FOLDS,
    ):
        X_train = X.iloc[train_indices].copy()
        X_valid = X.iloc[valid_indices].copy()
        y_train = y[train_indices]
        y_valid = y[valid_indices]
        X_test_copy = copy.deepcopy(X_test)

        # initiate model
        model = LGBMClassifier(**config.LGBM_PARAMS)

        # Training model
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            callbacks=[early_stopping(100), log_evaluation(period=1000)],
        )

        # Fold level predictions
        fold_oof_predictions = model.predict_proba(X_valid)
        fold_test_predictions = model.predict_proba(X_test_copy)

        # Updating global predictions
        oof_predictions[valid_indices] = fold_oof_predictions
        test_predictions += fold_test_predictions * test_weight

        # Inference and scoring
        fold_logloss = log_loss(y_valid, fold_oof_predictions)
        fold_preds = model.predict(X_valid)
        fold_bacc = balanced_accuracy_score(y_valid, fold_preds)

        fold_loglosses.append(fold_logloss)
        fold_baccs.append(fold_bacc)

        print(f"Fold: {fold} | LogLoss: {fold_logloss:.5f} | BACC: {fold_bacc:.5f}")

    # Calculating important metrics
    oof_logloss = log_loss(y, oof_predictions)
    oof_preds = model.classes_[np.argmax(oof_predictions, axis=1)]
    oof_bacc = balanced_accuracy_score(y, oof_preds)
    std_oof_logloss = np.std(fold_loglosses)
    std_oof_bacc = np.std(fold_baccs)

    # Mlflow metrics logging
    mlflow.log_metric("oof_logloss", oof_logloss)
    mlflow.log_metric("oof_bacc", oof_bacc)
    mlflow.log_metric("std_oof_logloss", std_oof_logloss)
    mlflow.log_metric("std_oof_bacc", std_oof_bacc)

    # Mlflow params logging
    mlflow.log_params(config.LGBM_PARAMS)

    # Garbage collector
    if config.N_FOLDS < 5:
        del X_train, y_train, X_valid, y_valid, model

# Saving oof predictions
oof_proba_df = pd.DataFrame(oof_predictions, columns=enc.classes_)
oof_proba_df.insert(0, "id", train_ids)
oof_path = Path(config.OOF_PROBA_DIR, f"{RUN_NAME}_oof_proba.csv")
oof_proba_df.to_csv(oof_path, index=False)

# Saving test predictions
test_proba_df = pd.DataFrame(test_predictions, columns=enc.classes_)
test_proba_df.insert(0, "id", test_ids)
test_path = Path(config.TEST_PROBA_DIR, f"{RUN_NAME}_test_proba.csv")
test_proba_df.to_csv(test_path, index=False)

Model Training:   0%|          | 0/5 [00:00<?, ?it/s]

(461877, 9)
9
Training until validation scores don't improve for 100 rounds
[1000]	valid_0's balanced_accuracy: 0.958334
[2000]	valid_0's balanced_accuracy: 0.960438
Early stopping, best iteration is:
[2377]	valid_0's balanced_accuracy: 0.960804


In [57]:
X_train

,_absolute_g_proxy,_absolute_r_proxy,_alpha_minus_delta,_alpha_mod1,_alpha_plus_delta,_balmer_break_proxy,_color_cross_gr_ri,_color_cross_ug_gr,_coord_x,_coord_y,_coord_z,_cos_alpha,_cos_delta,_delta_mod1,_dist_modulus_proxy,_g-r,_g_/_redshift,_galactic_b,_galactic_l,_i-z,_i_/_redshift,_is_star_redshift,_log_redshift,_r-i,_r_x_redshift,_sin_alpha,_sin_delta,_total_spatial_distance,_u-g,_u-z,_ugriz_kurt,_ugriz_max_diff,_ugriz_mean,_ugriz_skew,_ugriz_std,alpha,alpha_cat_,alpha_cat__delta_cat__,delta,delta_100_quantile_bin_,delta_500_quantile_bin_,delta_cat_,g,g_cat_,g_cat__r_cat__,g_r,galaxy_locus_dist,galaxy_population,i,i_cat_,i_z,orig_alpha_count,orig_alpha_mean,orig_alpha_median,orig_alpha_skew,orig_alpha_std,orig_delta_count,orig_delta_mean,orig_delta_median,orig_delta_skew,orig_delta_std,orig_g_count,orig_g_mean,orig_g_median,orig_g_skew,orig_g_std,orig_galaxy_population_count,orig_galaxy_population_mean,orig_galaxy_population_median,orig_galaxy_population_skew,orig_galaxy_population_std,orig_i_count,orig_i_mean,orig_i_median,orig_i_skew,orig_i_std,orig_r_count,orig_r_mean,orig_r_median,orig_r_skew,orig_r_std,orig_redshift_count,orig_redshift_mean,orig_redshift_median,orig_redshift_skew,orig_redshift_std,orig_spectral_type_count,orig_spectral_type_mean,orig_spectral_type_median,orig_spectral_type_skew,orig_spectral_type_std,orig_u_count,orig_u_mean,orig_u_median,orig_u_skew,orig_u_std,orig_z_count,orig_z_mean,orig_z_median,orig_z_skew,orig_z_std,qso_locus_dist,r,r_cat_,r_i,redshift,redshift_cat_,spectral_type,stellar_locus_dist,u,u_cat_,u_cat__z_cat__,u_g,z,z_cat_,_alpha_cat__delta_cat__TE_class0,_alpha_cat__delta_cat__TE_class1,_alpha_cat__delta_cat__TE_class2,_g_cat__r_cat__TE_class0,_g_cat__r_cat__TE_class1,_g_cat__r_cat__TE_class2,_u_cat__z_cat__TE_class0,_u_cat__z_cat__TE_class1,_u_cat__z_cat__TE_class2
0,3.212336,3.148427,130.774979,0.734256,164.693527,0.436819,1.692646,5.499440,-0.808809,0.510631,0.291692,-0.845581,0.956512,0.959273,-1.078910,1.537632,3.998871,47.192322,216.994690,0.636056,3.872977,0,0.342868,1.100813,2.232810,0.533847,0.291692,148.704498,3.576564,6.851065,1.217259,6.851065,21.120756,1.233607,2.731221,147.734256,147.0,147.0_16.0,16.959273,43,215,16.0,21.895559,21.0,21.0_20.0,1.537632,2.239139,Red_Sequence,19.257113,19.0,0.636056,962,0.359374,0.0,0.835214,0.527648,7.826044,0.375741,0.0,0.823684,0.568794,47164,0.624756,0.0,0.725016,0.739462,105213,0.365943,0.0,1.637377,0.749307,51217,0.419724,0.0,1.382336,0.715536,52918,0.518085,0.0,1.022671,0.724553,140565,0.280127,0.0,2.061866,0.653283,94290,0.257737,0.0,2.212969,0.652584,14244.0,0.326594,0.0,1.812878,0.699278,37257,0.543200,0.0,1.021074,0.837903,1.608694,20.357926,20.0,1.100813,0.342868,0.0,M,1.326446,25.472123,25.0,25.0_18.0,3.576564,18.621057,18.0,0.932295,0.056360,0.011281,0.625628,0.261294,0.113077,0.974170,0.002827,0.023002
1,3.181967,3.117695,95.641960,0.988678,160.335388,1.138713,0.541659,2.536923,-0.519995,0.665835,0.535041,-0.615506,0.844826,0.346716,-1.610844,1.499854,4.802560,34.281986,190.703995,0.439634,4.700860,0,0.146673,0.361141,1.329288,0.788132,0.535041,132.012924,1.691447,3.992076,-0.115831,3.992076,18.293056,1.032936,1.636652,127.988677,127.0,127.0_32.0,32.346716,66,331,32.0,19.087062,19.0,19.0_17.0,1.499854,0.822446,Red_Sequence,17.226067,17.0,0.439634,1111,0.501784,0.0,0.546436,0.607712,7.976252,0.473434,0.0,0.607721,0.595024,17907,0.979561,1.0,0.038729,0.845700,105213,0.365943,0.0,1.637377,0.749307,25134,0.598711,0.0,0.875941,0.901609,27937,0.534202,0.0,1.052598,0.874162,140565,0.280127,0.0,2.061866,0.653283,94290,0.257737,0.0,2.212969,0.652584,24849.0,0.814882,1.0,0.348006,0.806956,25684,0.495094,0.0,1.169881,0.857852,1.277424,17.587208,17.0,0.361141,0.146673,0.0,M,0.986137,20.778509,20.0,20.0_16.0,1.691447,16.786433,16.0,0.608431,0.229533,0.162022,0.962492,0.002216,0.035290,0.962585,0.001152,0.036262
2,2.986943,2.991608,144.447800,0.792648,215.137497,-0.681923,-0.054627,0.004072,-0.815680,0.002952,0.578496,-0.999993,0.81568

### XGBoost

In [ ]:
# Predictions
oof_predictions = np.zeros((len(train), config.N_CLASSES), dtype=np.float32)
test_predictions = np.zeros((len(X_test), config.N_CLASSES), dtype=np.float32)

# Experiment Setup
EXPERIMENT_NAME = "xgbm"
VERSION = "v11"
RUN_NAME = f"{EXPERIMENT_NAME}_{VERSION}_seed{config.SEED}"

mlflow.set_experiment(EXPERIMENT_NAME)

# CV
skf = StratifiedKFold(n_splits=config.N_FOLDS, shuffle=True, random_state=config.SEED)

fold_loglosses = []
fold_baccs = []

test_weight = 1 / config.N_FOLDS

with mlflow.start_run(run_name=RUN_NAME):
    for fold, (train_indices, valid_indices) in tqdm(
        enumerate(skf.split(X, y), start=1), desc="Model Training", total=config.N_FOLDS
    ):
        # Train and validation data
        X_train = X.iloc[train_indices]
        X_valid = X.iloc[valid_indices]

        y_train = y[train_indices]
        y_valid = y[valid_indices]

        model = XGBClassifier(**config.XGB_PARAMS)

        # Training model
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            verbose=1000
        )

        # Fold level predictions
        fold_oof_predictions = model.predict_proba(X_valid)
        fold_test_predictions = model.predict_proba(X_test)

        # Updating global predictions
        oof_predictions[valid_indices] = fold_oof_predictions
        test_predictions += fold_test_predictions * test_weight

        # Inference and scoring
        fold_logloss = log_loss(y_valid, fold_oof_predictions)
        fold_preds = model.predict(X_valid)

        fold_bacc = balanced_accuracy_score(y_valid, fold_preds)

        fold_loglosses.append(fold_logloss)
        fold_baccs.append(fold_bacc)

        print(f"Fold: {fold} | LogLoss: {fold_logloss:.5f} | BACC: {fold_bacc:.5f}")

    # Calculating important metrices
    oof_logloss = log_loss(y, oof_predictions)
    oof_preds = model.classes_[np.argmax(oof_predictions, axis=1)]
    oof_bacc = balanced_accuracy_score(y, oof_preds)
    std_oof_logloss = np.std(fold_loglosses)
    std_oof_bacc = np.std(fold_baccs)

    # Mlflow metrics logging
    mlflow.log_metric("oof_logloss", oof_logloss)
    mlflow.log_metric("oof_bacc", oof_bacc)
    mlflow.log_metric("std_oof_logloss", std_oof_logloss)
    mlflow.log_metric("std_oof_bacc", std_oof_bacc)

    # Mlflow params logging
    mlflow.log_params(config.LGBM_PARAMS)

    # Mlflow input feature names logging
    mlflow.log_dict({"feature_names": list(X_train.columns)}, "feature_names.json")

    # Garbase collector
    if config.N_FOLDS < 5:
        del X_train, y_train, X_valid, y_valid, model

# Saving oof predictions
oof_proba_df = pd.DataFrame(oof_predictions, columns=enc.classes_)
oof_proba_df.insert(0, "id", train_ids)

oof_path = Path(config.OOF_PROBA_DIR, f"{RUN_NAME}_oof_proba.csv")
oof_proba_df.to_csv(oof_path, index=False)

# Saving test predictions
test_proba_df = pd.DataFrame(test_predictions, columns=enc.classes_)
test_proba_df.insert(0, "id", test_ids)

test_path = Path(config.TEST_PROBA_DIR, f"{RUN_NAME}_test_proba.csv")
test_proba_df.to_csv(test_path, index=False)

In [ ]:
result = permutation_importance(
    model, X_valid, y_valid, n_repeats=5, random_state=42, n_jobs=-1
)

importance_df = pd.DataFrame(
    {"Feature": X.columns, "Importance_Drop": result.importances_mean}
).sort_values(by="Importance_Drop", ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(
    importance_df["Feature"], importance_df["Importance_Drop"], color="teal"
)
plt.xlabel("Drop in Validation Accuracy when Scrambled")
plt.title("Universal Feature Validity Audit")
plt.tight_layout()
plt.show()

### ResNet

In [ ]:
# Predictions
oof_predictions = np.zeros((len(train), config.N_CLASSES), dtype=np.float32)
test_predictions = np.zeros((len(X_test), config.N_CLASSES), dtype=np.float32)

# Experiment Setup
EXPERIMENT_NAME = "resnet"
VERSION = "v1"
RUN_NAME = f"{EXPERIMENT_NAME}_{VERSION}_seed{config.SEED}"

mlflow.set_experiment(EXPERIMENT_NAME)

# CV
skf = StratifiedKFold(n_splits=config.N_FOLDS, shuffle=True, random_state=config.SEED)

fold_loglosses = []
fold_baccs = []

test_weight = 1 / config.N_FOLDS

with mlflow.start_run(run_name=RUN_NAME):
    for fold, (train_indices, valid_indices) in tqdm(
        enumerate(skf.split(X, y), start=1), desc="Model Training", total=config.N_FOLDS
    ):
        # Train and validation data
        X_train = X.iloc[train_indices]
        X_valid = X.iloc[valid_indices]

        y_train = y[train_indices]
        y_valid = y[valid_indices]

        # Preprocessor
        # preprocess = ColumnTransformer(
        #    transformers=[
        #        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
        #    ], remainder='passthrough'
        # )

        model = Pipeline(
            steps=[
                # ("preprocess", preprocess),
                ("classifier", Resnet_RTDL_D_Classifier(**config.RESNET_PARAMS))
            ]
        )

        # Training model
        model.fit(
            X_train, 
            y_train, 
            classifier__X_val=X_valid,
            classifier__y_val=y_valid 
        )

        # Fold level predictions
        fold_oof_predictions = model.predict_proba(X_valid)
        fold_test_predictions = model.predict_proba(X_test)

        # Updating global predictions
        oof_predictions[valid_indices] = fold_oof_predictions
        test_predictions += fold_test_predictions * test_weight

        # Inference and scoring
        fold_logloss = log_loss(y_valid, fold_oof_predictions)
        fold_preds = model.predict(X_valid)

        fold_bacc = balanced_accuracy_score(y_valid, fold_preds)

        fold_loglosses.append(fold_logloss)
        fold_baccs.append(fold_bacc)

        print(f"Fold: {fold} | LogLoss: {fold_logloss:.5f} | BACC: {fold_bacc:.5f}")

    # Calculating important metrices
    oof_logloss = log_loss(y, oof_predictions)
    oof_preds = model.classes_[np.argmax(oof_predictions, axis=1)]
    oof_bacc = balanced_accuracy_score(y, oof_preds)
    std_oof_logloss = np.std(fold_loglosses)
    std_oof_bacc = np.std(fold_baccs)

    # Mlflow metrics logging
    mlflow.log_metric("oof_logloss", oof_logloss)
    mlflow.log_metric("oof_bacc", oof_bacc)
    mlflow.log_metric("std_oof_logloss", std_oof_logloss)
    mlflow.log_metric("std_oof_bacc", std_oof_bacc)

    # Mlflow params logging
    mlflow.log_params(config.RESNET_PARAMS)

    # Mlflow input feature names logging
    mlflow.log_dict({"feature_names": list(X_train.columns)}, "feature_names.json")

    # Mlflow preprocessing steps logging
    mlflow.log_text(str(model.steps), "steps.txt")

    # Garbase collector
    if config.N_FOLDS < 5:
        del X_train, y_train, X_valid, y_valid, model

# Saving oof predictions
oof_proba_df = pd.DataFrame(oof_predictions, columns=model.classes_)
oof_proba_df.insert(0, "id", train_ids)

oof_path = Path(config.OOF_PROBA_DIR, f"{RUN_NAME}_oof_proba.csv")
oof_proba_df.to_csv(oof_path, index=False)

# Saving test predictions
test_proba_df = pd.DataFrame(test_predictions, columns=model.classes_)
test_proba_df.insert(0, "id", test_ids)

test_path = Path(config.TEST_PROBA_DIR, f"{RUN_NAME}_test_proba.csv")
test_proba_df.to_csv(test_path, index=False)

### CatBoost

In [ ]:
# Predictions
oof_predictions = np.zeros((len(train), config.N_CLASSES), dtype=np.float32)
test_predictions = np.zeros((len(X_test), config.N_CLASSES), dtype=np.float32)

# Experiment Setup
EXPERIMENT_NAME = "catgbm"
VERSION = "v1"
RUN_NAME = f"{EXPERIMENT_NAME}_{VERSION}_seed{config.SEED}"

mlflow.set_experiment(EXPERIMENT_NAME)

# CV
skf = StratifiedKFold(n_splits=config.N_FOLDS, shuffle=True, random_state=config.SEED)

fold_loglosses = []
fold_baccs = []

test_weight = 1 / config.N_FOLDS

with mlflow.start_run(run_name=RUN_NAME):
    for fold, (train_indices, valid_indices) in tqdm(
        enumerate(skf.split(X, y), start=1), desc="Model Training", total=config.N_FOLDS
    ):
        # Train and validation data
        X_train = X.iloc[train_indices]
        X_valid = X.iloc[valid_indices]

        y_train = y[train_indices]
        y_valid = y[valid_indices]

        model = CatBoostClassifier(**config.CATBOOST_PARAMS)

        # Training model
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            verbose_eval=1000,
            cat_features=cat_cols
        )

        # Fold level predictions
        fold_oof_predictions = model.predict_proba(X_valid)
        fold_test_predictions = model.predict_proba(X_test)

        # Updating global predictions
        oof_predictions[valid_indices] = fold_oof_predictions
        test_predictions += fold_test_predictions * test_weight

        # Inference and scoring
        fold_logloss = log_loss(y_valid, fold_oof_predictions)
        fold_preds = model.predict(X_valid)

        fold_bacc = balanced_accuracy_score(y_valid, fold_preds)

        fold_loglosses.append(fold_logloss)
        fold_baccs.append(fold_bacc)

        print(f"Fold: {fold} | LogLoss: {fold_logloss:.5f} | BACC: {fold_bacc:.5f}")

    # Calculating important metrices
    oof_logloss = log_loss(y, oof_predictions)
    oof_preds = model.classes_[np.argmax(oof_predictions, axis=1)]
    oof_bacc = balanced_accuracy_score(y, oof_preds)
    std_oof_logloss = np.std(fold_loglosses)
    std_oof_bacc = np.std(fold_baccs)

    # Mlflow metrics logging
    mlflow.log_metric("oof_logloss", oof_logloss)
    mlflow.log_metric("oof_bacc", oof_bacc)
    mlflow.log_metric("std_oof_logloss", std_oof_logloss)
    mlflow.log_metric("std_oof_bacc", std_oof_bacc)

    # Mlflow params logging
    mlflow.log_params(config.CATBOOST_PARAMS)

    # Mlflow input feature names logging
    mlflow.log_dict({"feature_names": list(X_train.columns)}, "feature_names.json")

    # Garbase collector
    if config.N_FOLDS < 5:
        del X_train, y_train, X_valid, y_valid, model

# Saving oof predictions
oof_proba_df = pd.DataFrame(oof_predictions, columns=enc.classes_)
oof_proba_df.insert(0, "id", train_ids)

oof_path = Path(config.OOF_PROBA_DIR, f"{RUN_NAME}_oof_proba.csv")
oof_proba_df.to_csv(oof_path, index=False)

# Saving test predictions
test_proba_df = pd.DataFrame(test_predictions, columns=enc.classes_)
test_proba_df.insert(0, "id", test_ids)

test_path = Path(config.TEST_PROBA_DIR, f"{RUN_NAME}_test_proba.csv")
test_proba_df.to_csv(test_path, index=False)

### TABM

In [ ]:
# Predictions
oof_predictions = np.zeros((len(train), config.N_CLASSES), dtype=np.float32)
test_predictions = np.zeros((len(X_test), config.N_CLASSES), dtype=np.float32)

# Experiment Setup
EXPERIMENT_NAME = "tabm"
VERSION = "v1"
RUN_NAME = f"{EXPERIMENT_NAME}_{VERSION}_seed{config.SEED}"

mlflow.set_experiment(EXPERIMENT_NAME)

# CV
skf = StratifiedKFold(n_splits=config.N_FOLDS, shuffle=True, random_state=config.SEED)

fold_loglosses = []
fold_baccs = []

test_weight = 1 / config.N_FOLDS

with mlflow.start_run(run_name=RUN_NAME):
    for fold, (train_indices, valid_indices) in tqdm(
        enumerate(skf.split(X, y), start=1), desc="Model Training", total=config.N_FOLDS
    ):
        # Train and validation data
        X_train = X.iloc[train_indices]
        X_valid = X.iloc[valid_indices]

        y_train = y[train_indices]
        y_valid = y[valid_indices]

        X_train = pd.concat([X_train, X_orig], axis=0)
        y_train = np.concat([y_train, y_orig], axis=0)

        model = TabM_D_Classifier(**config.TABM_PARAMS)

        # Training model
        model.fit(
            X_train,
            y_train,
            X_valid,
            y_valid,
            cat_col_names=cat_cols
        )

        # Fold level predictions
        fold_oof_predictions = model.predict_proba(X_valid)
        fold_test_predictions = model.predict_proba(X_test)

        # Updating global predictions
        oof_predictions[valid_indices] = fold_oof_predictions
        test_predictions += fold_test_predictions * test_weight

        # Inference and scoring
        fold_logloss = log_loss(y_valid, fold_oof_predictions)
        fold_preds = model.predict(X_valid)

        fold_bacc = balanced_accuracy_score(y_valid, fold_preds)

        fold_loglosses.append(fold_logloss)
        fold_baccs.append(fold_bacc)

        print(f"Fold: {fold} | LogLoss: {fold_logloss:.5f} | BACC: {fold_bacc:.5f}")

    # Calculating important metrices
    oof_logloss = log_loss(y, oof_predictions)
    oof_preds = model.classes_[np.argmax(oof_predictions, axis=1)]
    oof_bacc = balanced_accuracy_score(y, oof_preds)
    std_oof_logloss = np.std(fold_loglosses)
    std_oof_bacc = np.std(fold_baccs)

    # Mlflow metrics logging
    mlflow.log_metric("oof_logloss", oof_logloss)
    mlflow.log_metric("oof_bacc", oof_bacc)
    mlflow.log_metric("std_oof_logloss", std_oof_logloss)
    mlflow.log_metric("std_oof_bacc", std_oof_bacc)

    # Mlflow params logging
    mlflow.log_params(config.TABM_PARAMS)

    # Mlflow input feature names logging
    mlflow.log_dict({"feature_names": list(X_train.columns)}, "feature_names.json")

    # Garbase collector
    del X_train, y_train, X_valid, y_valid, model

# Saving oof predictions
oof_proba_df = pd.DataFrame(oof_predictions, columns=config.PROB_COLS)
oof_proba_df.insert(0, "id", train_ids)

oof_path = Path(config.OOF_PROBA_DIR, f"{RUN_NAME}_oof_proba.csv")
oof_proba_df.to_csv(oof_path, index=False)

# Saving test predictions
test_proba_df = pd.DataFrame(test_predictions, columns=config.PROB_COLS)
test_proba_df.insert(0, "id", test_ids)

test_path = Path(config.TEST_PROBA_DIR, f"{RUN_NAME}_test_proba.csv")
test_proba_df.to_csv(test_path, index=False)

### Tabpfn

In [ ]:
# Predictions
oof_predictions = np.zeros((len(train), config.N_CLASSES), dtype=np.float32)

# --- SPLIT X_TEST INTO TWO BATCHES ---
half_idx = len(X_test) // 2
X_test_batch1 = X_test.iloc[:half_idx]
X_test_batch2 = X_test.iloc[half_idx:]

# Separate prediction arrays for each batch
test_predictions_b1 = np.zeros((len(X_test_batch1), config.N_CLASSES), dtype=np.float32)
test_predictions_b2 = np.zeros((len(X_test_batch2), config.N_CLASSES), dtype=np.float32)

# Experiment Setup
EXPERIMENT_NAME = "tabpfn"
VERSION = "v2"
RUN_NAME = f"{EXPERIMENT_NAME}_{VERSION}_seed{config.SEED}"

mlflow.set_experiment(EXPERIMENT_NAME)

# CV
skf = StratifiedKFold(n_splits=config.N_FOLDS, shuffle=True, random_state=config.SEED)

fold_loglosses = []
fold_baccs = []

test_weight = 1 / config.N_FOLDS

with mlflow.start_run(run_name=RUN_NAME):
    for fold, (train_indices, valid_indices) in tqdm(
        enumerate(skf.split(X, y), start=1), desc="Model Training", total=config.N_FOLDS
    ):
        # Train and validation data
        X_train = X.iloc[train_indices]
        X_valid = X.iloc[valid_indices]

        y_train = y[train_indices]
        y_valid = y[valid_indices]

        # Initiate model
        model = TabPFNClassifier(
            random_state=config.SEED,
            balance_probabilities=True
        )

        # Training model
        model.fit(
            X_train,
            y_train
        )

        # Fold level predictions
        fold_oof_predictions = model.predict_proba(X_valid)
        
        # Predict in two batches
        fold_test_predictions_b1 = model.predict_proba(X_test_batch1)
        fold_test_predictions_b2 = model.predict_proba(X_test_batch2)

        # Updating global predictions
        oof_predictions[valid_indices] = fold_oof_predictions
        
        test_predictions_b1 += fold_test_predictions_b1 * test_weight
        test_predictions_b2 += fold_test_predictions_b2 * test_weight

        # Inference and scoring
        fold_logloss = log_loss(y_valid, fold_oof_predictions)
        fold_preds = model.predict(X_valid)

        fold_bacc = balanced_accuracy_score(y_valid, fold_preds)

        fold_loglosses.append(fold_logloss)
        fold_baccs.append(fold_bacc)

        print(f"Fold: {fold} | LogLoss: {fold_logloss:.5f} | BACC: {fold_bacc:.5f}")

    # Calculating important metrices
    oof_logloss = log_loss(y, oof_predictions)
    oof_preds = model.classes_[np.argmax(oof_predictions, axis=1)]
    oof_bacc = balanced_accuracy_score(y, oof_preds)
    std_oof_logloss = np.std(fold_loglosses)
    std_oof_bacc = np.std(fold_baccs)

    # Mlflow metrics logging
    mlflow.log_metric("oof_logloss", oof_logloss)
    mlflow.log_metric("oof_bacc", oof_bacc)
    mlflow.log_metric("std_oof_logloss", std_oof_logloss)
    mlflow.log_metric("std_oof_bacc", std_oof_bacc)

    # Mlflow input feature names logging
    mlflow.log_dict({"feature_names": list(X_train.columns)}, "feature_names.json")

    # Garbase collector
    del X_train, y_train, X_valid, y_valid, model

# Saving oof predictions
oof_proba_df = pd.DataFrame(oof_predictions, columns=config.PROB_COLS)
oof_proba_df.insert(0, "id", train_ids)

oof_path = Path(config.OOF_PROBA_DIR, f"{RUN_NAME}_oof_proba.csv")
oof_proba_df.to_csv(oof_path, index=False)

test_predictions = np.vstack([test_predictions_b1, test_predictions_b2])

# Saving test predictions
test_proba_df = pd.DataFrame(test_predictions, columns=config.PROB_COLS)
test_proba_df.insert(0, "id", test_ids)

test_path = Path(config.TEST_PROBA_DIR, f"{RUN_NAME}_test_proba.csv")
test_proba_df.to_csv(test_path, index=False)

### AutoGluon

In [ ]:
# Predictions
oof_predictions = np.zeros((len(train), config.N_CLASSES), dtype=np.float32)
test_predictions = np.zeros((len(X_test), config.N_CLASSES), dtype=np.float32)

# Experiment Setup
EXPERIMENT_NAME = "autogluon"
VERSION = "v2"
RUN_NAME = f"{EXPERIMENT_NAME}_{VERSION}_seed{config.SEED}"

mlflow.set_experiment(EXPERIMENT_NAME)

# CV
skf = StratifiedKFold(n_splits=config.N_FOLDS, shuffle=True, random_state=config.SEED)

fold_loglosses = []
fold_baccs = []

test_weight = 1 / config.N_FOLDS

with mlflow.start_run(run_name=RUN_NAME):
    for fold, (train_indices, valid_indices) in tqdm(
        enumerate(skf.split(X, y), start=1), desc="Model Training", total=config.N_FOLDS
    ):
        # Train and validation data split
        X_train = X.iloc[train_indices].copy()
        X_valid = X.iloc[valid_indices].copy()

        y_train = y[train_indices]
        y_valid = y[valid_indices]

        # Combine features and target into a single DataFrame for AutoGluon
        train_data = X_train.copy()
        train_data['class'] = y_train

        valid_data = X_valid.copy()
        valid_data['class'] = y_valid

        fold_path = f"AutogluonModels/{EXPERIMENT_NAME}_{VERSION}_fold_{fold}"

        # 2. Initialize and train the TabularPredictor
        model = TabularPredictor(
            label='class', 
            problem_type='multiclass',
            eval_metric='balanced_accuracy',
            path=fold_path
        ).fit(
            train_data=train_data,
            tuning_data=valid_data,
            time_limit=300,
            presets='best_quality',
            refit_full=False
        )

        # Fold level predictions
        fold_oof_predictions = model.predict_proba(X_valid).values
        fold_test_predictions = model.predict_proba(X_test).values

        # Updating global predictions
        oof_predictions[valid_indices] = fold_oof_predictions
        test_predictions += fold_test_predictions * test_weight

        # Inference and scoring
        fold_logloss = log_loss(y_valid, fold_oof_predictions)
        fold_preds = model.predict(X_valid)

        fold_bacc = balanced_accuracy_score(y_valid, fold_preds)

        fold_loglosses.append(fold_logloss)
        fold_baccs.append(fold_bacc)

        print(f"Fold: {fold} | LogLoss: {fold_logloss:.5f} | BACC: {fold_bacc:.5f}")

    # Calculating important metrices
    oof_logloss = log_loss(y, oof_predictions)
    
    # Updated class extraction to handle AutoGluon's class alignment safely
    oof_preds = np.array(model.classes_)[np.argmax(oof_predictions, axis=1)]
    oof_bacc = balanced_accuracy_score(y, oof_preds)
    std_oof_logloss = np.std(fold_loglosses)
    std_oof_bacc = np.std(fold_baccs)

    # Mlflow metrics logging
    mlflow.log_metric("oof_logloss", oof_logloss)
    mlflow.log_metric("oof_bacc", oof_bacc)
    mlflow.log_metric("std_oof_logloss", std_oof_logloss)
    mlflow.log_metric("std_oof_bacc", std_oof_bacc)

    # Mlflow input feature names logging
    mlflow.log_dict({"feature_names": list(X_train.columns)}, "feature_names.json")

    # Garbage collector
    del X_train, y_train, X_valid, y_valid, train_data, valid_data, model

# Saving oof predictions
oof_proba_df = pd.DataFrame(oof_predictions, columns=config.PROB_COLS)
oof_proba_df.insert(0, "id", train_ids)

oof_path = Path(config.OOF_PROBA_DIR, f"{RUN_NAME}_oof_proba.csv")
oof_proba_df.to_csv(oof_path, index=False)

# Saving test predictions
test_proba_df = pd.DataFrame(test_predictions, columns=config.PROB_COLS)
test_proba_df.insert(0, "id", test_ids)

test_path = Path(config.TEST_PROBA_DIR, f"{RUN_NAME}_test_proba.csv")
test_proba_df.to_csv(test_path, index=False)